# 05 – Trainings-Dokumentation

Dieses Notebook dokumentiert den vollständigen Modell-Trainings-Workflow für die Bachelorarbeit.

**Struktur:**
1. **OOS-Datensatz erstellen** – wie die 100 ungesehenen Zeitreihen ausgewählt wurden
2. **Entfernte Features** – 7 redundante Features und Begründung
3. **Gemeinsame Infrastruktur** – Datenladen, Split, Scaler (für alle Modelle gleich)
4. **Klassische ML** – Linear Regression, Random Forest, LightGBM
5. **Deep Learning** – GRU, CNN-LSTM, TSMixer, iTransformer, Chronos (Zero-Shot)

> **Hinweis:** Der gesamte Training-Code wurde auf Kaggle (T4-GPU, 30 GB RAM) ausgeführt.
> Die Ergebnisse liegen als `.json` / `.npy` Dateien in `data/Kaggle_results/` vor.

## Teil 1: OOS-Datensatz erstellen

Vor dem Training wurden **100 Zeitreihen als Out-of-Sample (OOS) Testdatensatz** reserviert.
Diese Spots wurden **nie** für Training oder Hyperparameter-Suche verwendet.

**Auswahlprinzip:**
- Sortiere alle Spots nach Datensauberkeit (`clean = 1 - NaN/total`) und Länge
- Nimm die Top-406 als Trainings-Pool
- Von den verbleibenden Spots: Top-100 nach gleichen Kriterien als OOS-Set

Der Code liegt in `dataprep/unseen_dataset_builder.py` und wurde einmalig ausgeführt.

## Teil 2: Entfernte Features

Die folgenden Features wurden aus dem Feature-Set entfernt, da sie **linear mit anderen Features korrelieren**
und dadurch keine zusätzliche Erklärungskraft liefern:

| Feature | Grund |
|---------|-------|
| `ghi_magnitude` | Lineare Funktion von `ghi` (L2-Norm, redundant) |
| `temperature_penalty_rel` | Linear abgeleitet aus `temperature_2m` |
| `analog_kwh_1` | Skalierte Version von `scaled_analog_kwh_1` (redundant) |
| `analog_kwh_1_norm` | Normierte Version von `analog_kwh_1` (redundant) |
| `analog_knn_mean_norm` | Normierte Version von `scaled_analog_knn_mean` (redundant) |
| `analog_knn_mean` | Unskalierte Version von `scaled_analog_knn_mean` (redundant) |
| `analog_available` | Konstant >= 0.99 im Subset (keine Varianz, kein Informationsgewinn) |

Das bereinigte Feature-Set umfasst **45 Features** (siehe Zelle "Feature-Liste" unten).

In [ ]:
# ── OOS-Datensatz erstellen (einmalig ausgefuehrt, Ergebnis: data/processed/unseen_data.parquet) ──
# WARNUNG: Nur zur Dokumentation – muss nicht erneut ausgefuehrt werden!

import json, pyarrow.dataset as ds
from pathlib import Path

DATA_PATH = Path('data/processed/data_final.parquet')
TARGET = 'kwh_norm'

# Alle Spots analysieren
meta = ds.dataset(str(DATA_PATH)).to_table(columns=['spot_uuid', TARGET]).to_pandas()
stats = meta.groupby('spot_uuid', observed=True)[TARGET].agg(['count', lambda x: x.isna().sum()]).reset_index()
stats.columns = ['spot_uuid', 'total', 'nans']
stats['clean'] = 1 - (stats['nans'] / stats['total'])

# Die 406 fuer das Training genutzten Spots
selected = stats.sort_values(['clean', 'total'], ascending=False).iloc[:406]['spot_uuid'].tolist()

# Alle anderen Spots -- nach Sauberkeit und Vollstaendigkeit sortiert
unseen_stats = stats[~stats['spot_uuid'].isin(selected)].copy()
unseen_stats = unseen_stats.sort_values(['clean', 'total'], ascending=False)

# Top 100 sauberste und vollstaendigste ungesehene Spots
oos_spots = unseen_stats.iloc[:100]['spot_uuid'].tolist()

print(f'OOS Spots: {len(oos_spots)}')
print(f'Durchschnittliche Sauberkeit: {unseen_stats.iloc[:100]["clean"].mean():.4f}')
print(f'Durchschnittliche Laenge: {unseen_stats.iloc[:100]["total"].mean():.0f} Zeilen')

# Speichern fuer spaetere Verwendung auf Kaggle
with open('oos_spots.json', 'w') as f:
    json.dump(oos_spots, f)
print('oos_spots.json gespeichert')

## Teil 3: Gemeinsame Trainings-Infrastruktur

Alle 10 Modelle teilen denselben Datenladeblock, Split und Scaler.
Die folgenden Zellen dokumentieren diesen gemeinsamen Schritt.

> **Kaggle-Pfad:** `/kaggle/input/datasets/yashag03/data-final-parquet/data_final.parquet`

### Imports

Standard-Imports für das Kaggle-Notebook (T4-GPU, 30 GB RAM). Lokal nicht ausführbar.

In [ ]:
# ── Imports (Kaggle-Notebook) ────────────────────────────────────────────────
# WARNUNG: Dieser Block wurde auf Kaggle ausgefuehrt (T4 GPU, 30 GB RAM)
import gc, json, time, os
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.compute as pc
from pathlib import Path
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import optuna

DATA_PATH = Path('/kaggle/input/datasets/yashag03/data-final-parquet/data_final.parquet')
OOS_PATH  = Path('/kaggle/input/datasets/yashagrawal511/unseen/oos_spots.parquet')
TARGET    = 'kwh_norm'
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42); torch.cuda.manual_seed(42)
print(f'Device: {DEVICE}')

### Feature-Liste

In [ ]:
TARGET = 'kwh_norm'
FEATURES = ['ghi', 'dhi', 'dni', 'tsi', 'kt', 'dni_frac', 'dhi_frac',
            'solar_elevation', 'solar_azimuth', 'temperature_2m', 'precipitation_mm',
            'wind_speed_10m', 'relative_humidity_2m', 'snowfall', 'visibility',
            'weather_regime', 'snow_roll6h', 'snow_roll24h',
            'precip_roll6h', 'temp_roll24h', 'hour_sin', 'hour_cos', 'month_sin',
            'month_cos', 'doy_sin', 'doy_cos', 'latitude', 'longitude', 'panel_tilt',
            'max_kwh_est', 'kwp_est', 'panel_unimodal_azimuth', 'panel_tilt_azi_confidence',
            'panel_bimodal_east_azimuth', 'panel_bimodal_east_fraction',
            'panel_bimodal_west_azimuth', 'panel_is_ew_split', 'irr_mismatch',
            'scaled_analog_kwh_1', 'scaled_analog_knn_mean',
            'analog_dist_1', 'analog_dist_gap', 'analog_knn_std', 'analog_age_hours']
print(f'Anzahl Features: {len(FEATURES)}')

### Spot-Selektion (Top-406)

In [ ]:
# Waehlte die 406 Spots mit der besten Datensauberkeit und Laenge
print('Analysiere Spots...')
meta = ds.dataset(str(DATA_PATH)).to_table(columns=['spot_uuid', TARGET]).to_pandas()
stats = meta.groupby('spot_uuid', observed=True)[TARGET].agg(
    ['count', lambda x: x.isna().sum()]
).reset_index()
stats.columns = ['spot_uuid', 'total', 'nans']
stats['clean'] = 1 - (stats['nans'] / stats['total'])
selected = stats.sort_values(['clean', 'total'], ascending=False).iloc[:406]['spot_uuid'].tolist()
del meta, stats; gc.collect()
print(f'Ausgewaehlt: {len(selected)} Spots')

### Daten laden & bereinigen

Dreistufige Bereinigung:
1. **Nacht-Stunden (23:00-05:00)**: kwh\_norm explizit auf 0 setzen -- keine Solarproduktion möglich
2. **Ausreisser & verbleibende NaN**: kwh\_norm > 2.0 oder NaN (echte Lücken) werden
   mit `scaled_analog_knn_mean_norm` aufgefuellt -- dem starksten Einzelfeature
3. **Restliche NaN im Target**: gedroppt (kaum noch vorhanden nach Schritt 1+2)


In [ ]:
print('Lade Daten...')
df = ds.dataset(str(DATA_PATH)).to_table(
    columns=FEATURES + ['spot_uuid', TARGET, 'ts'],
    filter=pc.field('spot_uuid').isin(selected)
).to_pandas()
df['ts'] = pd.to_datetime(df['ts'])

# ── Schritt 1: Nacht-Stunden -> 0 ────────────────────────────────────────────
# Zwischen 23:00 und 05:00 ist Solarproduktion physikalisch 0
_night = (df['ts'].dt.hour >= 23) | (df['ts'].dt.hour < 5)
df.loc[_night, TARGET] = 0.0
print(f'Nacht-Stunden auf 0 gesetzt: {_night.sum():,} Zeilen')

# ── Schritt 2: Ausreisser & NaN -> scaled Analog-KNN-Norm ────────────────────
# kwh_norm > 2.0 = physikalisch unrealistisch (> 200% Nennleistung)
# NaN im Target = echte Messluecke (Tages-Zeros mit hohem kt aus data_final)
# Beide werden mit dem besten verfuegbaren Schaetzer ersetzt: scaled_analog_knn_mean_norm
_analog_fill = 'scaled_analog_knn_mean_norm'
if _analog_fill in df.columns:
    _bad = df[TARGET].isna() | (df[TARGET] > 2.0)
    _analog_vals = (df[_analog_fill].clip(0, 1.0).fillna(0.0))
    df.loc[_bad, TARGET] = _analog_vals[_bad]
    print(f'Ausreisser/NaN mit Analog-KNN gefuellt: {_bad.sum():,} Zeilen')
else:
    # Fallback falls Analog-Feature nicht verfuegbar
    df.loc[df[TARGET] > 2.0, TARGET] = np.nan
    print(f'Analog-Feature nicht gefunden -- Ausreisser auf NaN gesetzt')

# ── Schritt 3: Verbleibende NaN droppen (sollten minimal sein) ───────────────
n_before = len(df)
df = df.dropna(subset=[TARGET]).reset_index(drop=True)
print(f'Verbleibende NaN gedroppt: {n_before - len(df):,} Zeilen')

# ── Features: ffill + fillna(0) ──────────────────────────────────────────────
df[FEATURES] = df[FEATURES].ffill().fillna(0.0)
print(f'\nDatensatz final: {len(df):,} Zeilen | {df[TARGET].min():.4f} <= kwh_norm <= {df[TARGET].max():.4f}')

# Zeitbasierter Train/Val/Test-Split
train_end = pd.Timestamp('2025-10-31 23:00:00')
val_end   = pd.Timestamp('2025-12-31 23:00:00')
_split = np.zeros(len(df), dtype='int8')
_split[df['ts'] > train_end] = 1
_split[df['ts'] > val_end]   = 2
print(f'Train: {(_split==0).sum():,} | Val: {(_split==1).sum():,} | Test: {(_split==2).sum():,}')


### StandardScaler + Segment-Index

Für Sequenzmodelle wird ein `segments`-Index benötigt, der die Grenzen zwischen Spots markiert.
Der Scaler wird nur auf dem Trainingsset gefittet.

In [ ]:
def get_segments(df, mask):
    segs, curr = [], 0
    for _, g in df[mask].groupby('spot_uuid', observed=True):
        n = len(g); segs.append((curr, n)); curr += n
    return segs

segs_tr = get_segments(df, _split == 0)
segs_va = get_segments(df, _split == 1)
segs_te = get_segments(df, _split == 2)

scaler = StandardScaler()
X_tr = scaler.fit_transform(df.loc[_split==0, FEATURES].values.astype(np.float32))
y_tr = df.loc[_split==0, TARGET].values.astype(np.float32)
X_va = scaler.transform(df.loc[_split==1, FEATURES].values.astype(np.float32))
y_va = df.loc[_split==1, TARGET].values.astype(np.float32)
X_te = scaler.transform(df.loc[_split==2, FEATURES].values.astype(np.float32))
y_te = df.loc[_split==2, TARGET].values.astype(np.float32)
print(f'Train: {len(X_tr):,} | Val: {len(X_va):,} | Test: {len(X_te):,}')
del df; gc.collect()

## Teil 4: Klassische ML-Modelle

### 4.1 Linear Regression

Lineares Basismodell. Kein Sequenzkontext, nur Features des aktuellen Zeitpunkts.
Kein Hyperparameter-Tuning nötig (OLS-Loesung ist analytisch optimal).

Linear Regression löst analytisch (kein iteratives Training) -- daher keine Trainings-Kurve.

In [ ]:
# ── Linear Regression: Training und Evaluation ───────────────────────────────
# WARNUNG: Auf Kaggle ausgefuehrt
from sklearn.linear_model import LinearRegression
import joblib

t0 = time.time()
linreg_model = LinearRegression(n_jobs=-1)
linreg_model.fit(X_tr, y_tr)
y_p_linreg = np.maximum(0, linreg_model.predict(X_te))

linreg_mae  = float(mean_absolute_error(y_te, y_p_linreg))
linreg_rmse = float(np.sqrt(mean_squared_error(y_te, y_p_linreg)))
linreg_r2   = float(r2_score(y_te, y_p_linreg))
print(f'Test  --> MAE: {linreg_mae:.6f} | RMSE: {linreg_rmse:.6f} | R2: {linreg_r2:.6f}')

# OOS
oos_spots_list = pd.read_parquet(OOS_PATH)['spot_uuid'].unique().tolist()
X_oos_all = ds.dataset(str(DATA_PATH)).to_table(
    columns=FEATURES + ['spot_uuid', TARGET],
    filter=papc.field('spot_uuid').isin(oos_spots_list)
).to_pandas()
X_oos = X_oos_all[FEATURES].fillna(0).values.astype('float32')
y_oos = X_oos_all[TARGET].values.astype('float32')
sc_oos = scaler.transform(X_oos)
y_p_linreg_oos = np.maximum(0, linreg_model.predict(sc_oos))
oos_mae = float(mean_absolute_error(y_oos, y_p_linreg_oos))
oos_r2  = float(r2_score(y_oos, y_p_linreg_oos))
print(f'OOS   --> MAE: {oos_mae:.6f} | R2: {oos_r2:.6f}')
print(f'Zeit: {time.time()-t0:.1f}s')

joblib.dump(linreg_model, '/kaggle/working/linreg_model.pkl')
import json as _j
with open('/kaggle/working/linreg_results.json', 'w') as f:
    _j.dump({'mae': linreg_mae, 'rmse': linreg_rmse, 'r2': linreg_r2,
             'oos': {'mae': oos_mae, 'r2': oos_r2}}, f, indent=2)
print('Modell und Ergebnisse gespeichert.')

### 4.2 Random Forest

Ensemble-Modell aus unabhängig trainierten Entscheidungsbäumen.
Keine Trainings-Kurve (Baeume werden parallel gebaut, nicht iterativ optimiert).

Optuna-Suche: 25 Trials über `n_estimators`, `max_depth`, `max_features`, `min_samples_leaf`, `max_samples`.

In [ ]:
# ── Random Forest: Optuna ─────────────────────────────────────────────────────
# WARNUNG: Auf Kaggle ausgefuehrt
from sklearn.ensemble import RandomForestRegressor

def rf_objective(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 50, 200),
        max_depth=trial.suggest_int('max_depth', 15, 40),
        max_features=trial.suggest_float('max_features', 0.2, 0.8),
        min_samples_leaf=trial.suggest_int('min_samples_leaf', 10, 100),
        max_samples=trial.suggest_float('max_samples', 0.05, 0.15)
    )
    m = RandomForestRegressor(**params, n_jobs=-1, random_state=42)
    m.fit(X_tr[:2_000_000], y_tr[:2_000_000])  # Subset fuer schnelle Suche
    return mean_absolute_error(y_va, m.predict(X_va))

print('Starte Optuna (25 Trials)...')
rf_study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
rf_study.optimize(rf_objective, n_trials=25, show_progress_bar=True)
print(f'Beste Params: {rf_study.best_params} | Val-MAE: {rf_study.best_value:.6f}')

In [ ]:
# ── Random Forest: Finales Training und Evaluation ───────────────────────────
t0 = time.time()
rf_model = RandomForestRegressor(**rf_study.best_params, n_jobs=-1, random_state=42)
rf_model.fit(X_tr, y_tr)
y_p_rf = np.maximum(0, rf_model.predict(X_te))

rf_mae  = float(mean_absolute_error(y_te, y_p_rf))
rf_rmse = float(np.sqrt(mean_squared_error(y_te, y_p_rf)))
rf_r2   = float(r2_score(y_te, y_p_rf))
print(f'Test  --> MAE: {rf_mae:.6f} | RMSE: {rf_rmse:.6f} | R2: {rf_r2:.6f}')

# Feature Importance
fi_rf = dict(sorted(
    zip(FEATURES, [float(x) for x in rf_model.feature_importances_]),
    key=lambda x: x[1], reverse=True
))
print('Top-5 Features:', list(fi_rf.keys())[:5])

# OOS
X_oos_rf = df_oos[FEATURES].values.astype(np.float32)
y_oos_rf  = df_oos[TARGET].values.astype(np.float32)
y_oos_p_rf = np.maximum(0, rf_model.predict(X_oos_rf))
print(f'OOS   --> MAE: {mean_absolute_error(y_oos_rf, y_oos_p_rf):.6f} | '
      f'RMSE: {np.sqrt(mean_squared_error(y_oos_rf, y_oos_p_rf)):.6f} | '
      f'R2: {r2_score(y_oos_rf, y_oos_p_rf):.6f}')

joblib.dump(rf_model, 'rf_model.pkl')

### 4.3 LightGBM

Gradient-Boosting-Modell (sequentiell aufgebaute Baeume).
Unterstuetzt Early Stopping mit Validierungs-Feedback.

Optuna-Suche: 25 Trials über `learning_rate`, `num_leaves`, `min_child_samples`, `feature_fraction`, `bagging_fraction`, `reg_alpha`, `reg_lambda`.

In [ ]:
# ── LightGBM: Optuna ─────────────────────────────────────────────────────────
# WARNUNG: Auf Kaggle ausgefuehrt
import lightgbm as lgb

def lgbm_objective(trial):
    params = {
        'objective': 'regression', 'metric': 'mae', 'verbosity': -1, 'num_threads': -1,
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1),
        'num_leaves':        trial.suggest_int('num_leaves', 31, 255),
        'min_child_samples': trial.suggest_int('min_child_samples', 50, 200),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.6, 0.9),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.6, 0.9),
        'bagging_freq':      5,
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True),
    }
    dtr  = lgb.Dataset(X_tr[:2_000_000], label=y_tr[:2_000_000])
    dval = lgb.Dataset(X_va, label=y_va)
    m = lgb.train(params, dtr, num_boost_round=500, valid_sets=[dval],
                  callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    return mean_absolute_error(y_va, m.predict(X_va))

print('Starte Optuna (25 Trials)...')
lgbm_study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
lgbm_study.optimize(lgbm_objective, n_trials=25, show_progress_bar=True)
print(f'Beste Params: {lgbm_study.best_params} | Val-MAE: {lgbm_study.best_value:.6f}')

In [ ]:
# ── LightGBM: Finales Training und Evaluation ─────────────────────────────────
t0 = time.time()
lgbm_best = {**lgbm_study.best_params, 'objective': 'regression', 'metric': 'mae',
             'verbosity': -1, 'num_threads': -1, 'bagging_freq': 5}
dtr_full = lgb.Dataset(X_tr, label=y_tr)
dval     = lgb.Dataset(X_va, label=y_va)
lgbm_model = lgb.train(
    lgbm_best, dtr_full, num_boost_round=3000, valid_sets=[dval],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)]
)

y_p_lgbm = np.maximum(0, lgbm_model.predict(X_te))
lgbm_mae  = float(mean_absolute_error(y_te, y_p_lgbm))
lgbm_rmse = float(np.sqrt(mean_squared_error(y_te, y_p_lgbm)))
lgbm_r2   = float(r2_score(y_te, y_p_lgbm))
print(f'Test  --> MAE: {lgbm_mae:.6f} | RMSE: {lgbm_rmse:.6f} | R2: {lgbm_r2:.6f}')

fi_lgbm = dict(sorted(
    zip(FEATURES, lgbm_model.feature_importance(importance_type='gain')),
    key=lambda x: x[1], reverse=True
))
print('Top-5 Features:', list(fi_lgbm.keys())[:5])

# OOS
X_oos_lgbm = scaler.transform(df_oos[FEATURES].values.astype(np.float32))
y_oos_lgbm = df_oos[TARGET].values.astype(np.float32)
y_oos_p_lgbm = np.maximum(0, lgbm_model.predict(X_oos_lgbm))
print(f'OOS   --> MAE: {mean_absolute_error(y_oos_lgbm, y_oos_p_lgbm):.6f} | '
      f'RMSE: {np.sqrt(mean_squared_error(y_oos_lgbm, y_oos_p_lgbm)):.6f} | '
      f'R2: {r2_score(y_oos_lgbm, y_oos_p_lgbm):.6f}')

lgbm_model.save_model('lgbm_model.txt')

## Teil 5: Deep Learning Modelle

### Gemeinsame DL-Infrastruktur

Alle sequenzbasierten Modelle (GRU, CNN-LSTM, TSMixer, iTransformer) verwenden
dieselbe `SpotSeqDS` und `extract_targets` Implementierung.

In [ ]:
class SpotSeqDS(Dataset):
    def __init__(self, X, y, segs, slen, stride):
        self.X, self.y, self.slen = X, y, slen
        self.starts = []
        for off, n in segs:
            if n >= slen:
                for i in range(0, n - slen + 1, stride):
                    self.starts.append(off + i)
    def __len__(self): return len(self.starts)
    def __getitem__(self, i):
        s = self.starts[i]
        return torch.from_numpy(self.X[s:s+self.slen]), torch.tensor(self.y[s+self.slen-1])

def extract_targets(y, segs, slen, stride=1):
    y_t, pos = [], 0
    for off, n in segs:
        if n >= slen:
            for i in range(0, n - slen + 1, stride):
                y_t.append(y[pos + slen - 1 + i])
        pos += n
    return np.array(y_t)

print('SpotSeqDS und extract_targets definiert.')

### 5.1 GRU

Gated Recurrent Unit -- klassisches Sequenzmodell.
Optuna-Suche: 25 Trials über `slen`, `hid`, `n_layers`, `dropout`, `lr`.
Für jede Testsequenz wird der letzte Hidden-State als Vorhersage verwendet.

In [ ]:
# ── GRU: Modell-Architektur ───────────────────────────────────────────────────
class GRUMod(nn.Module):
    def __init__(self, inp, hid, n_layers, dropout):
        super().__init__()
        self.gru = nn.GRU(inp, hid, num_layers=n_layers, batch_first=True,
                          dropout=dropout if n_layers > 1 else 0.0)
        self.fc = nn.Sequential(nn.Linear(hid, hid // 2), nn.ReLU(), nn.Linear(hid // 2, 1))
    def forward(self, x):
        _, h = self.gru(x)
        return self.fc(h[-1]).squeeze()

print('GRUMod definiert.')

In [ ]:
# ── GRU: Optuna-Suche (25 Trials) ────────────────────────────────────────────
# WARNUNG: Auf Kaggle ausgefuehrt (T4 GPU)
def gru_objective(trial):
    try:
        slen     = trial.suggest_int('slen', 96, 576, step=96)
        hid      = trial.suggest_int('hid', 64, 256)
        n_layers = trial.suggest_int('n_layers', 1, 3)
        dropout  = trial.suggest_float('dropout', 0.1, 0.4)
        lr       = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
        m   = GRUMod(len(FEATURES), hid, n_layers, dropout).to(DEVICE)
        opt = torch.optim.Adam(m.parameters(), lr=lr)
        dl  = DataLoader(SpotSeqDS(X_tr, y_tr, segs_tr, slen, stride=96),
                         batch_size=256, shuffle=True, pin_memory=True, num_workers=2)
        dva = DataLoader(SpotSeqDS(X_va, y_va, segs_va, slen, stride=slen),
                         batch_size=512, pin_memory=True, num_workers=2)
        best_val, no_improve = float('inf'), 0
        for ep in range(15):
            m.train()
            for xb, yb in dl:
                opt.zero_grad()
                nn.L1Loss()(m(xb.to(DEVICE)), yb.to(DEVICE)).backward()
                opt.step()
            m.eval(); val_loss = 0.0
            with torch.no_grad():
                for xb, yb in dva:
                    val_loss += nn.L1Loss()(m(xb.to(DEVICE)), yb.to(DEVICE)).item() * len(xb)
            val_loss /= len(dva.dataset)
            if val_loss < best_val - 1e-5: best_val = val_loss; no_improve = 0
            else:
                no_improve += 1
                if no_improve >= 3: break
        return best_val
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); gc.collect(); return float('inf')

gru_study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
gru_study.optimize(gru_objective, n_trials=25, show_progress_bar=True)
print(f'Beste Params: {gru_study.best_params} | Val-MAE: {gru_study.best_value:.6f}')

In [ ]:
# ── GRU: Finales Training (50 Epochs, Early Stopping) ────────────────────────
gru_best = gru_study.best_params
gru_model = GRUMod(len(FEATURES), gru_best['hid'], gru_best['n_layers'], gru_best['dropout']).to(DEVICE)
gru_opt = torch.optim.Adam(gru_model.parameters(), lr=gru_best['lr'])
gru_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(gru_opt, patience=3, factor=0.5)
train_dl_gru = DataLoader(SpotSeqDS(X_tr, y_tr, segs_tr, gru_best['slen'], stride=48),
                          batch_size=256, shuffle=True, pin_memory=True, num_workers=2)
val_dl_gru   = DataLoader(SpotSeqDS(X_va, y_va, segs_va, gru_best['slen'], stride=gru_best['slen']),
                          batch_size=512, pin_memory=True, num_workers=2)

gru_best_val, gru_no_improve, gru_state = float('inf'), 0, None
gru_train_losses, gru_val_losses = [], []
t0 = time.time()

for ep in range(50):
    gru_model.train(); ep_loss, ep_n = 0.0, 0
    for xb, yb in train_dl_gru:
        gru_opt.zero_grad()
        loss = nn.L1Loss()(gru_model(xb.to(DEVICE)), yb.to(DEVICE))
        loss.backward(); gru_opt.step()
        ep_loss += loss.item() * len(xb); ep_n += len(xb)
    gru_train_losses.append(ep_loss / ep_n)
    gru_model.eval(); val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_dl_gru:
            val_loss += nn.L1Loss()(gru_model(xb.to(DEVICE)), yb.to(DEVICE)).item() * len(xb)
    val_loss /= len(val_dl_gru.dataset); gru_val_losses.append(val_loss)
    gru_sched.step(val_loss)
    print(f'Epoch {ep+1:02d} | Train: {gru_train_losses[-1]:.6f} | Val: {val_loss:.6f}')
    if val_loss < gru_best_val - 1e-5:
        gru_best_val = val_loss; gru_no_improve = 0
        gru_state = {k: v.cpu().clone() for k, v in gru_model.state_dict().items()}
    else:
        gru_no_improve += 1
        if gru_no_improve >= 5: print(f'Early stopping Epoch {ep+1}'); break

gru_model.load_state_dict(gru_state)
print(f'Training abgeschlossen. Zeit: {time.time()-t0:.0f}s')

In [ ]:
# ── GRU: Test + OOS Evaluation ────────────────────────────────────────────────
gru_model.eval()
test_dl_gru = DataLoader(SpotSeqDS(X_te, y_te, segs_te, gru_best['slen'], stride=1),
                         batch_size=512, pin_memory=True, num_workers=2)
preds = []
with torch.no_grad():
    for xb, _ in test_dl_gru: preds.append(gru_model(xb.to(DEVICE)).cpu().numpy())
y_p_gru = np.maximum(0, np.concatenate(preds))
y_t_gru = extract_targets(y_te, segs_te, gru_best['slen'], stride=1)
print(f'Test  --> MAE: {mean_absolute_error(y_t_gru, y_p_gru):.6f} | '
      f'RMSE: {np.sqrt(mean_squared_error(y_t_gru, y_p_gru)):.6f} | '
      f'R2: {r2_score(y_t_gru, y_p_gru):.6f}')

# OOS
df_oos_gru = ds.dataset(str(DATA_PATH)).to_table(
    columns=FEATURES + ['spot_uuid', TARGET],
    filter=pc.field('spot_uuid').isin(oos_spots_list)
).to_pandas().dropna(subset=[TARGET]).reset_index(drop=True)
df_oos_gru[FEATURES] = df_oos_gru[FEATURES].ffill().fillna(0.0)
segs_oos_gru, curr = [], 0
for _, g in df_oos_gru.groupby('spot_uuid', observed=True):
    n = len(g); segs_oos_gru.append((curr, n)); curr += n
X_oos_gru = scaler.transform(df_oos_gru[FEATURES].values.astype(np.float32))
y_oos_gru = df_oos_gru[TARGET].values.astype(np.float32)
del df_oos_gru; gc.collect()

oos_dl_gru = DataLoader(SpotSeqDS(X_oos_gru, y_oos_gru, segs_oos_gru, gru_best['slen'], stride=gru_best['slen']),
                        batch_size=512, pin_memory=True, num_workers=2)
preds_oos = []
with torch.no_grad():
    for xb, _ in oos_dl_gru: preds_oos.append(gru_model(xb.to(DEVICE)).cpu().numpy())
y_oos_p_gru = np.maximum(0, np.concatenate(preds_oos))
y_oos_t_gru = extract_targets(y_oos_gru, segs_oos_gru, gru_best['slen'], stride=gru_best['slen'])
print(f'OOS   --> MAE: {mean_absolute_error(y_oos_t_gru, y_oos_p_gru):.6f} | '
      f'RMSE: {np.sqrt(mean_squared_error(y_oos_t_gru, y_oos_p_gru)):.6f} | '
      f'R2: {r2_score(y_oos_t_gru, y_oos_p_gru):.6f}')

torch.save(gru_model.state_dict(), 'gru_model.pt')

### 5.2 CNN-LSTM

1D-CNN zur lokalen Feature-Extraktion gefolgt von LSTM für zeitliche Abhängigkeiten.
Hyperparameter wurden vorab per Optuna gefunden und hier als feste Werte gesetzt.

Bestes Ergebnis: `slen=480, cnn_ch=52, hid=99, n_layers=1, dropout=0.191, lr=0.00112`

In [ ]:
# ── CNN-LSTM: Modell-Architektur ──────────────────────────────────────────────
class CNNLSTMMod(nn.Module):
    def __init__(self, inp, cnn_ch, hid, n_layers, dropout):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(inp, cnn_ch, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_ch), nn.ReLU()
        )
        self.lstm = nn.LSTM(cnn_ch, hid, num_layers=n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0.0)
        self.fc = nn.Sequential(nn.Linear(hid, hid // 2), nn.ReLU(), nn.Linear(hid // 2, 1))
    def forward(self, x):
        x = self.conv(x.permute(0, 2, 1)).permute(0, 2, 1)
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1]).squeeze()

cnnlstm_best = {
    'slen': 480, 'cnn_ch': 52, 'hid': 99, 'n_layers': 1,
    'dropout': 0.19127267288786132, 'lr': 0.0011207606211860567
}
print('CNN-LSTM Konfiguration geladen.')

In [ ]:
# ── CNN-LSTM: Training (50 Epochs, Early Stopping) ───────────────────────────
cnnlstm = CNNLSTMMod(len(FEATURES), cnnlstm_best['cnn_ch'], cnnlstm_best['hid'],
                     cnnlstm_best['n_layers'], cnnlstm_best['dropout']).to(DEVICE)
cnnlstm_opt   = torch.optim.Adam(cnnlstm.parameters(), lr=cnnlstm_best['lr'])
cnnlstm_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(cnnlstm_opt, patience=3, factor=0.5)
train_dl_cnn = DataLoader(SpotSeqDS(X_tr, y_tr, segs_tr, cnnlstm_best['slen'], stride=48),
                          batch_size=256, shuffle=True, pin_memory=True, num_workers=2)
val_dl_cnn   = DataLoader(SpotSeqDS(X_va, y_va, segs_va, cnnlstm_best['slen'], stride=cnnlstm_best['slen']),
                          batch_size=512, pin_memory=True, num_workers=2)

cnn_best_val, cnn_no_imp, cnn_state = float('inf'), 0, None
cnn_train_losses, cnn_val_losses = [], []
t0 = time.time()

for ep in range(50):
    cnnlstm.train(); ep_loss, ep_n = 0.0, 0
    for xb, yb in train_dl_cnn:
        cnnlstm_opt.zero_grad()
        loss = nn.L1Loss()(cnnlstm(xb.to(DEVICE)), yb.to(DEVICE))
        loss.backward(); cnnlstm_opt.step()
        ep_loss += loss.item() * len(xb); ep_n += len(xb)
    cnn_train_losses.append(ep_loss / ep_n)
    cnnlstm.eval(); val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_dl_cnn:
            val_loss += nn.L1Loss()(cnnlstm(xb.to(DEVICE)), yb.to(DEVICE)).item() * len(xb)
    val_loss /= len(val_dl_cnn.dataset); cnn_val_losses.append(val_loss)
    cnnlstm_sched.step(val_loss)
    print(f'Epoch {ep+1:02d} | Train: {cnn_train_losses[-1]:.6f} | Val: {val_loss:.6f}')
    if val_loss < cnn_best_val - 1e-5:
        cnn_best_val = val_loss; cnn_no_imp = 0
        cnn_state = {k: v.cpu().clone() for k, v in cnnlstm.state_dict().items()}
    else:
        cnn_no_imp += 1
        if cnn_no_imp >= 5: print(f'Early stopping Epoch {ep+1}'); break

cnnlstm.load_state_dict(cnn_state)
print(f'Training abgeschlossen. Zeit: {time.time()-t0:.0f}s')

In [ ]:
# ── CNN-LSTM: Test + OOS Evaluation ──────────────────────────────────────────
cnnlstm.eval()
test_dl_cnn = DataLoader(SpotSeqDS(X_te, y_te, segs_te, cnnlstm_best['slen'], stride=1),
                         batch_size=512, pin_memory=True, num_workers=2)
preds = []
with torch.no_grad():
    for xb, _ in test_dl_cnn: preds.append(cnnlstm(xb.to(DEVICE)).cpu().numpy())
y_p_cnn = np.maximum(0, np.concatenate(preds))
y_t_cnn = extract_targets(y_te, segs_te, cnnlstm_best['slen'], stride=1)
print(f'len(y_p)={len(y_p_cnn)} | len(y_t)={len(y_t_cnn)} | Gleich={len(y_p_cnn)==len(y_t_cnn)}')
print(f'Test  --> MAE: {mean_absolute_error(y_t_cnn, y_p_cnn):.6f} | '
      f'RMSE: {np.sqrt(mean_squared_error(y_t_cnn, y_p_cnn)):.6f} | '
      f'R2: {r2_score(y_t_cnn, y_p_cnn):.6f}')

torch.save(cnnlstm.state_dict(), 'cnnlstm_model.pt')

### 5.3 TSMixer

MLP-Mixer Architektur für Zeitreihen: alternierende Time-Mixing und Feature-Mixing MLP-Schichten.
Kein Attention-Mechanismus, reine MLP-Architektur.

Optuna-Suche: 25 Trials über `slen`, `d_model`, `n_blocks`, `dropout`, `lr`.

In [ ]:
# ── TSMixer: Modell-Architektur ───────────────────────────────────────────────
class TSMixerBlock(nn.Module):
    def __init__(self, seq_len, n_features, dropout):
        super().__init__()
        self.norm1    = nn.LayerNorm(n_features)
        self.norm2    = nn.LayerNorm(n_features)
        self.time_mix = nn.Sequential(nn.Linear(seq_len, seq_len), nn.GELU(), nn.Dropout(dropout))
        self.feat_mix = nn.Sequential(nn.Linear(n_features, n_features), nn.GELU(), nn.Dropout(dropout))
    def forward(self, x):
        residual = x; x = self.norm1(x)
        x = self.time_mix(x.transpose(1, 2)).transpose(1, 2) + residual
        residual = x; x = self.norm2(x)
        x = self.feat_mix(x) + residual
        return x

class TSMixer(nn.Module):
    def __init__(self, seq_len, n_features, d_model, n_blocks, dropout):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.blocks     = nn.ModuleList([TSMixerBlock(seq_len, d_model, dropout) for _ in range(n_blocks)])
        self.norm       = nn.LayerNorm(d_model)
        self.head       = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model // 2, 1)
        )
    def forward(self, x):
        x = self.input_proj(x)
        for block in self.blocks: x = block(x)
        return self.head(self.norm(x)[:, -1, :]).squeeze(-1)

print('TSMixer definiert.')

In [ ]:
# ── TSMixer: Optuna-Suche (25 Trials) + Training ─────────────────────────────
# WARNUNG: Auf Kaggle ausgefuehrt
def tsmixer_objective(trial):
    try:
        slen     = trial.suggest_int('slen', 96, 576, step=96)
        d_model  = trial.suggest_categorical('d_model', [32, 64, 128])
        n_blocks = trial.suggest_int('n_blocks', 1, 4)
        dropout  = trial.suggest_float('dropout', 0.0, 0.3)
        lr       = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
        m   = TSMixer(slen, len(FEATURES), d_model, n_blocks, dropout).to(DEVICE)
        opt = torch.optim.Adam(m.parameters(), lr=lr)
        dl  = DataLoader(SpotSeqDS(X_tr, y_tr, segs_tr, slen, stride=96),
                         batch_size=256, shuffle=True, pin_memory=True, num_workers=2)
        dva = DataLoader(SpotSeqDS(X_va, y_va, segs_va, slen, stride=slen),
                         batch_size=512, pin_memory=True, num_workers=2)
        best_val, no_improve = float('inf'), 0
        for ep in range(15):
            m.train()
            for xb, yb in dl:
                opt.zero_grad()
                nn.L1Loss()(m(xb.to(DEVICE)), yb.to(DEVICE)).backward()
                opt.step()
            m.eval(); val_loss = 0.0
            with torch.no_grad():
                for xb, yb in dva:
                    val_loss += nn.L1Loss()(m(xb.to(DEVICE)), yb.to(DEVICE)).item() * len(xb)
            val_loss /= len(dva.dataset)
            if val_loss < best_val - 1e-5: best_val = val_loss; no_improve = 0
            else:
                no_improve += 1
                if no_improve >= 3: break
        return best_val
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); gc.collect(); return float('inf')

tsmixer_study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
tsmixer_study.optimize(tsmixer_objective, n_trials=25, show_progress_bar=True)
print(f'Beste Params: {tsmixer_study.best_params} | Val-MAE: {tsmixer_study.best_value:.6f}')

In [ ]:
# ── TSMixer: Finales Training ─────────────────────────────────────────────────
tsm_best = tsmixer_study.best_params
tsm_model = TSMixer(tsm_best['slen'], len(FEATURES), tsm_best['d_model'],
                    tsm_best['n_blocks'], tsm_best['dropout']).to(DEVICE)
tsm_opt   = torch.optim.Adam(tsm_model.parameters(), lr=tsm_best['lr'])
tsm_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(tsm_opt, patience=3, factor=0.5)
train_dl_tsm = DataLoader(SpotSeqDS(X_tr, y_tr, segs_tr, tsm_best['slen'], stride=48),
                          batch_size=256, shuffle=True, pin_memory=True, num_workers=2)
val_dl_tsm   = DataLoader(SpotSeqDS(X_va, y_va, segs_va, tsm_best['slen'], stride=tsm_best['slen']),
                          batch_size=512, pin_memory=True, num_workers=2)

tsm_best_val, tsm_no_imp, tsm_state = float('inf'), 0, None
tsm_train_losses, tsm_val_losses = [], []
t0 = time.time()

for ep in range(50):
    tsm_model.train(); ep_loss, ep_n = 0.0, 0
    for xb, yb in train_dl_tsm:
        tsm_opt.zero_grad()
        loss = nn.L1Loss()(tsm_model(xb.to(DEVICE)), yb.to(DEVICE))
        loss.backward(); tsm_opt.step()
        ep_loss += loss.item() * len(xb); ep_n += len(xb)
    tsm_train_losses.append(ep_loss / ep_n)
    tsm_model.eval(); val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_dl_tsm:
            val_loss += nn.L1Loss()(tsm_model(xb.to(DEVICE)), yb.to(DEVICE)).item() * len(xb)
    val_loss /= len(val_dl_tsm.dataset); tsm_val_losses.append(val_loss)
    tsm_sched.step(val_loss)
    print(f'Epoch {ep+1:02d} | Train: {tsm_train_losses[-1]:.6f} | Val: {val_loss:.6f}')
    if val_loss < tsm_best_val - 1e-5:
        tsm_best_val = val_loss; tsm_no_imp = 0
        tsm_state = {k: v.cpu().clone() for k, v in tsm_model.state_dict().items()}
    else:
        tsm_no_imp += 1
        if tsm_no_imp >= 5: print(f'Early stopping Epoch {ep+1}'); break

tsm_model.load_state_dict(tsm_state)
print(f'Training abgeschlossen. Zeit: {time.time()-t0:.0f}s')

# Evaluation
tsm_model.eval()
test_dl_tsm = DataLoader(SpotSeqDS(X_te, y_te, segs_te, tsm_best['slen'], stride=1),
                         batch_size=512, pin_memory=True, num_workers=2)
preds = []
with torch.no_grad():
    for xb, _ in test_dl_tsm: preds.append(tsm_model(xb.to(DEVICE)).cpu().numpy())
y_p_tsm = np.maximum(0, np.concatenate(preds))
y_t_tsm = extract_targets(y_te, segs_te, tsm_best['slen'], stride=1)
print(f'Test  --> MAE: {mean_absolute_error(y_t_tsm, y_p_tsm):.6f} | '
      f'RMSE: {np.sqrt(mean_squared_error(y_t_tsm, y_p_tsm)):.6f} | '
      f'R2: {r2_score(y_t_tsm, y_p_tsm):.6f}')
torch.save(tsm_model.state_dict(), 'tsmixer_model.pt')

### 5.4 iTransformer

Inverted Transformer: Self-Attention wird über die Feature-Dimension angewendet, nicht über Zeit.
Jedes Feature bekommt einen eigenen Token (Zeitreihe als Embedding).

Optuna-Suche: 25 Trials über `slen`, `d_model`, `n_heads`, `n_layers`, `d_ff`, `dropout`, `lr`.

In [ ]:
# ── iTransformer: Modell-Architektur ─────────────────────────────────────────
N_FEATURES = len(FEATURES)

class ITransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    def forward(self, x):
        residual = x; x, _ = self.attn(x, x, x)
        x = self.norm1(x + residual)
        x = self.norm2(self.ff(x) + x)
        return x

class ITransformer(nn.Module):
    def __init__(self, seq_len, n_features, d_model, n_heads, n_layers, d_ff, dropout):
        super().__init__()
        self.input_proj = nn.Linear(seq_len, d_model)
        self.layers     = nn.ModuleList(
            [ITransformerEncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(n_features * d_model, d_model), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model, 1)
        )
    def forward(self, x):
        x = x.transpose(1, 2)       # (B, F, T)
        x = self.input_proj(x)      # (B, F, d_model)
        for layer in self.layers: x = layer(x)
        x = self.norm(x).flatten(1) # (B, F*d_model)
        return self.head(x).squeeze(-1)

print('iTransformer definiert.')

In [ ]:
# ── iTransformer: Optuna-Suche (25 Trials) ───────────────────────────────────
# WARNUNG: Auf Kaggle ausgefuehrt
def itx_objective(trial):
    try:
        slen     = trial.suggest_categorical('slen', [48, 96, 168, 336])
        d_model  = trial.suggest_categorical('d_model', [32, 64, 128])
        n_heads  = trial.suggest_categorical('n_heads', [2, 4, 8])
        n_layers = trial.suggest_int('n_layers', 1, 4)
        d_ff     = trial.suggest_categorical('d_ff', [64, 128, 256])
        dropout  = trial.suggest_float('dropout', 0.0, 0.3)
        lr       = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
        if d_model % n_heads != 0: return float('inf')
        m   = ITransformer(slen, N_FEATURES, d_model, n_heads, n_layers, d_ff, dropout).to(DEVICE)
        opt = torch.optim.Adam(m.parameters(), lr=lr)
        dl  = DataLoader(SpotSeqDS(X_tr, y_tr, segs_tr, slen, stride=96),
                         batch_size=512, shuffle=True, pin_memory=True, num_workers=2)
        dva = DataLoader(SpotSeqDS(X_va, y_va, segs_va, slen, stride=slen),
                         batch_size=1024, pin_memory=True, num_workers=2)
        best_val, no_improve = float('inf'), 0
        for ep in range(12):
            m.train()
            for xb, yb in dl:
                opt.zero_grad()
                nn.L1Loss()(m(xb.to(DEVICE)), yb.to(DEVICE)).backward()
                opt.step()
            m.eval(); val_loss = 0.0
            with torch.no_grad():
                for xb, yb in dva:
                    val_loss += nn.L1Loss()(m(xb.to(DEVICE)), yb.to(DEVICE)).item() * len(xb)
            val_loss /= len(dva.dataset)
            if val_loss < best_val - 1e-5: best_val = val_loss; no_improve = 0
            else:
                no_improve += 1
                if no_improve >= 3: break
        del m; torch.cuda.empty_cache(); gc.collect()
        return best_val
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); gc.collect(); return float('inf')

itx_study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
itx_study.optimize(itx_objective, n_trials=25, show_progress_bar=True)
print(f'Beste Params: {itx_study.best_params} | Val-MAE: {itx_study.best_value:.6f}')

In [ ]:
# ── iTransformer: Finales Training + Evaluation ───────────────────────────────
itx_best = itx_study.best_params
itx_model = ITransformer(itx_best['slen'], N_FEATURES, itx_best['d_model'],
                          itx_best['n_heads'], itx_best['n_layers'],
                          itx_best['d_ff'], itx_best['dropout']).to(DEVICE)
print(f'Parameter: {sum(p.numel() for p in itx_model.parameters()):,}')
itx_opt   = torch.optim.Adam(itx_model.parameters(), lr=itx_best['lr'])
itx_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(itx_opt, patience=3, factor=0.5)
train_dl_itx = DataLoader(SpotSeqDS(X_tr, y_tr, segs_tr, itx_best['slen'], stride=48),
                          batch_size=512, shuffle=True, pin_memory=True, num_workers=2)
val_dl_itx   = DataLoader(SpotSeqDS(X_va, y_va, segs_va, itx_best['slen'], stride=itx_best['slen']),
                          batch_size=1024, pin_memory=True, num_workers=2)

itx_best_val, itx_no_imp, itx_state = float('inf'), 0, None
itx_train_losses, itx_val_losses = [], []
t0 = time.time()

for ep in range(50):
    itx_model.train(); ep_loss, ep_n = 0.0, 0
    for xb, yb in train_dl_itx:
        itx_opt.zero_grad()
        loss = nn.L1Loss()(itx_model(xb.to(DEVICE)), yb.to(DEVICE))
        loss.backward(); itx_opt.step()
        ep_loss += loss.item() * len(xb); ep_n += len(xb)
    itx_train_losses.append(ep_loss / ep_n)
    itx_model.eval(); val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_dl_itx:
            val_loss += nn.L1Loss()(itx_model(xb.to(DEVICE)), yb.to(DEVICE)).item() * len(xb)
    val_loss /= len(val_dl_itx.dataset); itx_val_losses.append(val_loss)
    itx_sched.step(val_loss)
    print(f'Epoch {ep+1:02d} | Train MAE: {itx_train_losses[-1]:.6f} | Val MAE: {val_loss:.6f}')
    if val_loss < itx_best_val - 1e-5:
        itx_best_val = val_loss; itx_no_imp = 0
        itx_state = {k: v.cpu().clone() for k, v in itx_model.state_dict().items()}
    else:
        itx_no_imp += 1
        if itx_no_imp >= 5: print(f'Early stopping Epoch {ep+1}'); break

itx_model.load_state_dict(itx_state)
itx_model.eval()
test_dl_itx = DataLoader(SpotSeqDS(X_te, y_te, segs_te, itx_best['slen'], stride=1),
                         batch_size=1024, pin_memory=True, num_workers=2)
preds = []
with torch.no_grad():
    for xb, _ in test_dl_itx: preds.append(itx_model(xb.to(DEVICE)).cpu().numpy())
y_p_itx = np.maximum(0, np.concatenate(preds))
y_t_itx = extract_targets(y_te, segs_te, itx_best['slen'], stride=1)
print(f'Test  --> MAE: {mean_absolute_error(y_t_itx, y_p_itx):.6f} | '
      f'RMSE: {np.sqrt(mean_squared_error(y_t_itx, y_p_itx)):.6f} | '
      f'R2: {r2_score(y_t_itx, y_p_itx):.6f}')
torch.save(itx_model.state_dict(), 'itransformer_model.pt')

### 5.5 Chronos T5-Tiny (Zero-Shot)

Pre-trained Foundation Model von Amazon für Zeitreihenvorhersage.
**Zero-Shot:** kein Fine-Tuning, nur univariate `kwh_norm`-Historie als Kontext.

- Context-Länge: 512 Zeitschritte (= 128 Stunden, ca. 5.3 Tage)
- Prediction-Length: 144 Zeitschritte (= 36 Stunden, 15-min-Auflösung)
- Median über 20 stochastische Samples

Chronos benötigt **kein Feature-Engineering** -- der Vergleich zeigt, wie viel Mehrwert
die 45 Features der anderen Modelle gegenüber purem Zeitreihenverlauf bringen.

In [ ]:
# ── Chronos: Installation (Kaggle) ───────────────────────────────────────────
# WARNUNG: Auf Kaggle ausgefuehrt
import subprocess
subprocess.run(['pip', 'install', '-q',
    'git+https://github.com/amazon-science/chronos-forecasting.git',
    '--force-reinstall', '--no-deps'
], check=False)
subprocess.run(['pip', 'install', '-q', 'chronos-forecasting'], check=False)

In [ ]:
# ── Chronos: Zero-Shot Evaluation ────────────────────────────────────────────
from chronos import ChronosPipeline
from tqdm.auto import tqdm

CONTEXT_LEN = 512
PRED_LEN    = 144
BATCH_SIZE  = 8

# Nur kwh_norm wird geladen (kein Feature-Engineering)
df_chr = ds.dataset(str(DATA_PATH)).to_table(
    columns=['kwh_norm', 'spot_uuid', 'ts'],
    filter=pc.field('spot_uuid').isin(selected)
).to_pandas().dropna(subset=[TARGET]).reset_index(drop=True)
df_chr = df_chr[df_chr[TARGET] <= 2.0].reset_index(drop=True)
df_chr[TARGET] = df_chr[TARGET].ffill().fillna(0.0)

_split_chr = np.zeros(len(df_chr), dtype='int8')
_split_chr[pd.to_datetime(df_chr['ts']) > train_end] = 1
_split_chr[pd.to_datetime(df_chr['ts']) > val_end]   = 2

segs_te_chr = []
curr = 0
for _, g in df_chr[_split_chr==2].groupby('spot_uuid', observed=True):
    n = len(g); segs_te_chr.append((curr, n)); curr += n
y_te_chr = df_chr.loc[_split_chr==2, TARGET].values.astype(np.float32)
del df_chr; gc.collect()

print('Lade Chronos-T5-tiny (Zero-Shot)...')
pipeline = ChronosPipeline.from_pretrained(
    'amazon/chronos-t5-tiny',
    device_map=str(DEVICE),
    torch_dtype=torch.float16
)
print(f'Parameter: {sum(p.numel() for p in pipeline.model.parameters()):,}')

In [ ]:
# ── Chronos: Inference auf Testset ───────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader

class ChronosSeqDS(Dataset):
    def __init__(self, y, segs, context_len, pred_len, stride):
        self.y = y
        self.context_len = context_len
        self.pred_len = pred_len
        self.starts = []
        for off, n in segs:
            if n >= context_len + pred_len:
                for i in range(0, n - context_len - pred_len + 1, stride):
                    self.starts.append(off + i)
    def __len__(self): return len(self.starts)
    def __getitem__(self, i):
        s = self.starts[i]
        context = torch.tensor(self.y[s:s+self.context_len], dtype=torch.float32)
        target  = torch.tensor(self.y[s+self.context_len:s+self.context_len+self.pred_len],
                                dtype=torch.float32)
        return context, target

pipeline.model.eval()
test_ds_chr = ChronosSeqDS(y_te_chr, segs_te_chr, CONTEXT_LEN, PRED_LEN, stride=PRED_LEN)
test_dl_chr = DataLoader(test_ds_chr, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'Test Batches: {len(test_dl_chr)}')

preds_chr, targets_chr = [], []
with torch.no_grad():
    for context, target in tqdm(test_dl_chr, desc='Chronos Inference'):
        try:
            samples = pipeline.predict(context, prediction_length=PRED_LEN, num_samples=5)
            pred = samples.median(dim=1).values
            preds_chr.append(pred.numpy())
            targets_chr.append(target.numpy())
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); gc.collect()
            print('OOM -- Batch uebersprungen'); continue

y_p_chr = np.maximum(0, np.concatenate(preds_chr).flatten())
y_t_chr = np.concatenate(targets_chr).flatten()
print(f'Test  --> MAE: {mean_absolute_error(y_t_chr, y_p_chr):.6f} | '
      f'RMSE: {np.sqrt(mean_squared_error(y_t_chr, y_p_chr)):.6f} | '
      f'R2: {r2_score(y_t_chr, y_p_chr):.6f}')

## Zusammenfassung

Alle Ergebnisse (`.json`) und Vorhersagen (`.npy`) wurden als Kaggle-Outputs gespeichert
und liegen lokal unter `data/Kaggle_results/` in Unterordnern pro Modell.

Die vollständige Auswertung und der Vergleich aller Modelle erfolgt in `03_model_training.ipynb`.